In [14]:
# Import necessary libraries
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Subset, DataLoader, Dataset
from torchvision import datasets, transforms, models
from torchvision.models import efficientnet_b0
from torchvision.models.efficientnet import SqueezeExcitation
import matplotlib.pyplot as plt
import multiprocessing
import numpy as np
from sklearn.model_selection import StratifiedKFold, StratifiedShuffleSplit
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,  roc_auc_score, 
    cohen_kappa_score, matthews_corrcoef, confusion_matrix, average_precision_score, fbeta_score,
    precision_recall_curve, balanced_accuracy_score)
import pandas as pd
from sklearn.utils import shuffle
import random
import albumentations as A
from albumentations.pytorch import ToTensorV2
import pandas as pd
import torch_optimizer
import cv2
from skimage import measure
from torchvision.utils import make_grid
%matplotlib inline

In [15]:
# Check for GPU availability
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cuda


In [16]:
!nvidia-smi

Wed Dec 18 13:53:25 2024       
+---------------------------------------------------------------------------------------+
| NVIDIA-SMI 535.54.03              Driver Version: 535.54.03    CUDA Version: 12.2     |
|-----------------------------------------+----------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id        Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |         Memory-Usage | GPU-Util  Compute M. |
|                                         |                      |               MIG M. |
|=========================================+======================+======================|
|   0  Tesla V100-PCIE-32GB           On  | 00000000:3B:00.0 Off |                    0 |
| N/A   33C    P0              40W / 250W |   3290MiB / 32768MiB |      0%      Default |
|                                         |                      |                  N/A |
+-----------------------------------------+----------------------+--

In [17]:
file_name = "Benchmark_GoogLeNet_CV"


In [18]:
import os

# Define directories of where the datasets reside
base_dir = '/home/pxk409/skin_lesions_13k'

# Define directories of where the training, validation and test sets reside
train_dir = os.path.join(base_dir, 'train')
test_dir = os.path.join(base_dir, 'test')

# Define directories of where the benign and malignant images reside
train_benign_dir = os.path.join(base_dir, 'train/benign')
train_malignant_dir = os.path.join(base_dir, 'train/malignant')
test_benign_dir = os.path.join(base_dir, 'test/benign')
test_malignant_dir = os.path.join(base_dir, 'test/malignant')

In [19]:
# Let's check the number of images in each set
print('Total training benign images:', len(os.listdir(train_benign_dir)))
print('Total training malignant images:', len(os.listdir(train_malignant_dir)))
print('Total test benign images:', len(os.listdir(test_benign_dir)))
print('Total test malignant images:', len(os.listdir(test_malignant_dir)))

Total training benign images: 5000
Total training malignant images: 5000
Total test benign images: 1500
Total test malignant images: 1500


In [20]:
# Load full dataset
full_dataset = datasets.ImageFolder(root=train_dir)

# Extract labels from the dataset
#labels = [label for _, label in full_dataset.samples]

# Stratified split (80% train, 20% validation)
#split = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=123456)
#train_idx, val_idx = next(split.split(np.zeros(len(labels)), labels))

# Create training and validation subsets
#train_dataset = Subset(full_dataset, train_idx)
#val_dataset = Subset(full_dataset, val_idx)

#print(f"Training set: {len(train_idx)} samples")
#print(f"Validation set: {len(val_idx)} samples")

# Count the number of samples per class in each split
#train_classes = [full_dataset.targets[idx] for idx in train_dataset.indices]
#val_classes = [full_dataset.targets[idx] for idx in val_dataset.indices]

#print(f"Training set class counts: {np.bincount(train_classes)}")
#print(f"Validation set class counts: {np.bincount(val_classes)}")

In [21]:
# Custom transform to integrate Albumentations with PyTorch
class AlbumentationsTransform:
    def __init__(self, augmentations):
        self.augmentations = A.Compose(augmentations)

    def __call__(self, image):
        image = np.array(image)  # Convert PIL Image to NumPy array
        augmented = self.augmentations(image=image)
        return augmented["image"]

# Simple augmentations for training
simple_transform = AlbumentationsTransform([
    A.RandomResizedCrop(height=224, width=224, scale=(0.8, 1.0), p=1.0),  # Random crop and resize
    A.HorizontalFlip(p=0.5),  # Horizontal flip
    A.VerticalFlip(p=0.5),  # Vertical flip
    #A.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1, p=0.5),  # Color jitter
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),  # Normalize to match pre-trained model
    ToTensorV2(),  # Convert to PyTorch tensor and ensure dtype=torch.float32
])


# Use AlbumentationsTransform directly in the loaders for consistency
train_transform = simple_transform

# Albumentations augmentations for validation and test
albumentations_val_test_transform = AlbumentationsTransform([
    A.Resize(height=224, width=224),  # Resize to EfficientNet input size
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),  # Normalize to match pre-trained model
    ToTensorV2()  # Convert to PyTorch tensor
])

test_val_transform = albumentations_val_test_transform


In [22]:
class TransformDataset(Dataset):
    def __init__(self, dataset, transform):
        """
        Args:
            dataset: Original dataset (e.g., Subset of ImageFolder).
            transform: Transform to be applied to the images in the dataset.
        """
        self.dataset = dataset
        self.transform = transform

    def __getitem__(self, idx):
        image, label = self.dataset[idx]
        if self.transform:
            image = self.transform(image)
        return image, label

    def __len__(self):
        return len(self.dataset)

In [23]:
# Wrap subsets with the respective transforms
#train_dataset = TransformDataset(train_dataset, train_transform)
#val_dataset = TransformDataset(val_dataset, test_val_transform)

In [24]:
# Load and prepare the test dataset
#test_dataset = datasets.ImageFolder(root=test_dir, transform=test_val_transform)
test_dataset = datasets.ImageFolder(root=test_dir)
test_dataset = TransformDataset(test_dataset, test_val_transform)

In [25]:
# Create data loaders
batch_size = 64
#num_cpus = multiprocessing.cpu_count()  # Use all CPU cores
#print(f"Number of CPU cores: {num_cpus}")
#num_workers = 16


#train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
#val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)
#train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=num_workers, pin_memory=True)
#val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=num_workers, pin_memory=True)
#test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=num_workers, pin_memory=True)

In [26]:
def initialize_model():
    # Load pre-trained GoogLeNet 
    model = models.googlenet(weights=models.GoogLeNet_Weights.DEFAULT)

    # Disable auxiliary classifiers by removing or ignoring their outputs
    model.aux_logits = False  # This disables the aux_logits attribute

    # Replace the main classifier (fully connected layers) for binary classification
    model.fc = nn.Sequential(
        nn.Linear(model.fc.in_features, 512),
        nn.ReLU(),
        nn.Dropout(0.5),
        nn.Linear(512, 256),
        nn.ReLU(),
        nn.Dropout(0.5),
        nn.Linear(256, 2)
    )

    # Freeze the convolutional part
    for param in model.parameters():
        param.requires_grad = False

    # Unfreeze the classifier for training
    for param in model.fc.parameters():
        param.requires_grad = True

    # Move model to device
    model = model.to(device)
    return model

model = initialize_model()

In [27]:
# Function to count trainable and non-trainable parameters
def count_trainable_non_trainable_parameters(model):
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    non_trainable = sum(p.numel() for p in model.parameters() if not p.requires_grad)
    return trainable, non_trainable

# Count parameters in InceptionV3's feature extractor (convolutional part)
def count_googlenet_parts(model):
    # Count parameters in the main feature extraction layers
    print("Counting trainable/non-trainable parameters in InceptionV3...")

    # Count parameters in the main feature layers (excluding AuxLogits)
    trainable_main, non_trainable_main = count_trainable_non_trainable_parameters(model)

    # If aux_logits exists, count it separately
    if hasattr(model, 'AuxLogits') and model.AuxLogits is not None:
        aux_trainable, aux_non_trainable = count_trainable_non_trainable_parameters(model.AuxLogits)
        print(f"Auxiliary Classifier (AuxLogits):\n"
              f"  Trainable parameters: {aux_trainable:,}\n"
              f"  Non-trainable parameters: {aux_non_trainable:,}")
    else:
        aux_trainable, aux_non_trainable = 0, 0
        print("Auxiliary Classifier (AuxLogits): Not present or disabled.")

    # Classifier part (fully connected layers at the end)
    trainable_classifier, non_trainable_classifier = count_trainable_non_trainable_parameters(model.fc)

    print(f"Main Feature Extractor (without AuxLogits):\n"
          f"  Trainable parameters: {trainable_main - aux_trainable - trainable_classifier:,}\n"
          f"  Non-trainable parameters: {non_trainable_main - aux_non_trainable - non_trainable_classifier:,}")

    print(f"Final Classifier (fc layer):\n"
          f"  Trainable parameters: {trainable_classifier:,}\n"
          f"  Non-trainable parameters: {non_trainable_classifier:,}")

    print(f"Whole Model:\n"
          f"  Total Trainable parameters: {trainable_main:,}\n"
          f"  Total Non-trainable parameters: {non_trainable_main:,}")


# Call function to count parameters
count_googlenet_parts(model)


Counting trainable/non-trainable parameters in InceptionV3...
Auxiliary Classifier (AuxLogits): Not present or disabled.
Main Feature Extractor (without AuxLogits):
  Trainable parameters: 0
  Non-trainable parameters: 5,599,904
Final Classifier (fc layer):
  Trainable parameters: 656,642
  Non-trainable parameters: 0
Whole Model:
  Total Trainable parameters: 656,642
  Total Non-trainable parameters: 5,599,904


In [28]:
# Early Stopping
class EarlyStopping:
    def __init__(self, patience, delta, path):
        """
        Args:
            patience (int): Number of epochs to wait after last improvement.
            delta (float): Minimum change in monitored metric to qualify as an improvement.
            path (str): Path to save the best model.
        """
        self.patience = patience
        self.delta = delta
        self.path = path
        self.counter = 0
        self.best_score = None
        self.early_stop = False

    def __call__(self, val_loss, model):
        score = -val_loss  # Early stopping is based on minimizing validation loss

        if self.best_score is None:
            self.best_score = score
            self.save_checkpoint(val_loss, model)
        elif score < self.best_score + self.delta:
            self.counter += 1
            print(f"EarlyStopping counter: {self.counter} out of {self.patience}")
            if self.counter >= self.patience:
                self.early_stop = True
        else:
            self.best_score = score
            self.save_checkpoint(val_loss, model)
            self.counter = 0

    def save_checkpoint(self, val_loss, model):
        """Saves model when validation loss decreases."""
        print(f"Validation loss decreased. Saving model to {self.path}")
        torch.save(model.state_dict(), self.path)


# Set the path for saving best_model.pth
#current_dir = os.getcwd()
#model_save_path = os.path.join(current_dir, 'best_model2.pth')

# Initialize EarlyStopping
#early_stopping = EarlyStopping(patience=5, delta=0.001, path=model_save_path)

In [29]:
# Helper function to evaluate a dataset
def evaluate_model(model, data_loader, device):
    model.eval()  # Set model to evaluation mode
    all_labels = []
    all_preds = []
    all_probs = []

    with torch.no_grad():
        for images, labels in data_loader:
            images, labels = images.to(device), labels.to(device)

            # Forward pass
            outputs = model(images)
            probs = torch.softmax(outputs, dim=1)[:, 1]  # Probabilities for the positive class
            _, preds = torch.max(outputs, 1)

            all_labels.extend(labels.cpu().numpy())
            all_preds.extend(preds.cpu().numpy())
            all_probs.extend(probs.cpu().numpy())

    return np.array(all_labels), np.array(all_preds), np.array(all_probs)

# Helper function to calculate selected metrics
def calculate_metrics(labels, preds, probs):
    accuracy = accuracy_score(labels, preds)
    precision = precision_score(labels, preds)
    recall = recall_score(labels, preds)
    f1 = f1_score(labels, preds)
    f2 = fbeta_score(labels, preds, beta=2)
    auc = roc_auc_score(labels, probs)
    specificity = recall_score(labels, preds, pos_label=0)
    npv = precision_score(labels, preds, pos_label=0)

    return recall, precision, accuracy, f1, f2, auc, specificity, npv

In [30]:
# Training loop with 5-fold cross-validation
cv_results = []
cv_metrics = []
n_splits = 5
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=123456)

# Extract labels from the dataset
labels = [label for _, label in full_dataset.samples]

# Cross-validation loop
for fold, (train_idx, val_idx) in enumerate(skf.split(np.zeros(len(labels)), labels), 1):
    print(f"\n========== Fold {fold} ==========\n")
    
    # Create training and validation subsets for this fold
    train_dataset = Subset(full_dataset, train_idx)
    val_dataset = Subset(full_dataset, val_idx)
    
    # Apply transformations
    train_dataset = TransformDataset(train_dataset, train_transform)
    val_dataset = TransformDataset(val_dataset, test_val_transform)
    
    # Create data loaders
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
    
    # Initialize model and optimizer for each fold
    model = initialize_model()
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=0.001)

    # EarlyStopping initialization
    early_stopping = EarlyStopping(patience=10, delta=0.0001, path=os.path.join(os.getcwd(), f"{file_name}_fold_{fold}.pth"))

    # Training loop
    num_epochs = 100
    train_losses = []
    val_losses = []
    
    for epoch in range(num_epochs):
        # Training phase
        model.train()
        train_loss = 0
        train_correct = 0
        
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            
            # Forward pass
            outputs = model(images)
            loss = criterion(outputs, labels)
            
            # Backward pass and optimization
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            
            train_loss += loss.item()
            _, preds = torch.max(outputs, 1)
            train_correct += torch.sum(preds == labels)
        
        train_loss /= len(train_loader)
        train_accuracy = train_correct.double() / len(train_dataset)
        train_losses.append(train_loss)
        
        # Validation phase
        model.eval()
        val_loss = 0
        val_correct = 0
        
        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(device), labels.to(device)
                
                outputs = model(images)
                loss = criterion(outputs, labels)
                
                val_loss += loss.item()
                _, preds = torch.max(outputs, 1)
                val_correct += torch.sum(preds == labels)
        
        val_loss /= len(val_loader)
        val_accuracy = val_correct.double() / len(val_dataset)
        val_losses.append(val_loss)
        
        print(f"Epoch {epoch + 1}/{num_epochs} | "
              f"Train Loss: {train_loss:.4f}, Train Acc: {train_accuracy:.4f} | "
              f"Val Loss: {val_loss:.4f}, Val Acc: {val_accuracy:.4f}")
        
        # Call EarlyStopping
        early_stopping(val_loss, model)
        if early_stopping.early_stop:
            print("Early stopping triggered. Ending training.")
            break
    
    # Store results for this fold
    cv_results.append({
        "fold": fold,
        "train_loss": train_loss,
        "train_accuracy": train_accuracy.item(),
        "val_loss": val_loss,
        "val_accuracy": val_accuracy.item()
        })
        
    # Load the best model for evaluation
    model.load_state_dict(torch.load(f"{file_name}_fold_{fold}.pth", weights_only=True))

    # Evaluate on validation set
    labels, preds, probs = evaluate_model(model, val_loader, device)
    metrics = calculate_metrics(labels, preds, probs)
    recall, precision, accuracy, f1, f2, auc, specificity, npv = metrics

    # Record metrics for the fold
    cv_metrics.append({
        "Fold": fold,
        "Recall": recall,
        "Precision": precision,
        "Accuracy": accuracy,
        "F1 Score": f1,
        "F2 Score": f2,
        "AUC": auc,
        "Specificity": specificity,
        "NPV": npv,
    })

# Summarize cross-validation results
df_cv_results = pd.DataFrame(cv_results)
print("\n========== Cross-Validation Results ==========\n")
print(df_cv_results)

print("\nAverage Validation Accuracy: {:.4f}".format(df_cv_results["val_accuracy"].mean()))


========== Fold 1 ==========

Epoch 1/100 | Train Loss: 0.6003, Train Acc: 0.6764 | Val Loss: 0.5206, Val Acc: 0.7415
Validation loss decreased. Saving model to /home/pxk409/Skin_EfficientNet/Benchmark_GoogLeNet_CV_fold_1.pth
Epoch 2/100 | Train Loss: 0.5444, Train Acc: 0.7233 | Val Loss: 0.5210, Val Acc: 0.7495
EarlyStopping counter: 1 out of 10
Epoch 3/100 | Train Loss: 0.5343, Train Acc: 0.7313 | Val Loss: 0.5051, Val Acc: 0.7550
Validation loss decreased. Saving model to /home/pxk409/Skin_EfficientNet/Benchmark_GoogLeNet_CV_fold_1.pth
Epoch 4/100 | Train Loss: 0.5256, Train Acc: 0.7356 | Val Loss: 0.4942, Val Acc: 0.7635
Validation loss decreased. Saving model to /home/pxk409/Skin_EfficientNet/Benchmark_GoogLeNet_CV_fold_1.pth
Epoch 5/100 | Train Loss: 0.5176, Train Acc: 0.7398 | Val Loss: 0.4942, Val Acc: 0.7590
EarlyStopping counter: 1 out of 10
Epoch 6/100 | Train Loss: 0.5115, Train Acc: 0.7508 | Val Loss: 0.4815, Val Acc: 0.7660
Validation loss decreased. Saving model to /hom

Epoch 57/100 | Train Loss: 0.4304, Train Acc: 0.7924 | Val Loss: 0.4300, Val Acc: 0.8040
Validation loss decreased. Saving model to /home/pxk409/Skin_EfficientNet/Benchmark_GoogLeNet_CV_fold_1.pth
Epoch 58/100 | Train Loss: 0.4432, Train Acc: 0.7880 | Val Loss: 0.4328, Val Acc: 0.7920
EarlyStopping counter: 1 out of 10
Epoch 59/100 | Train Loss: 0.4299, Train Acc: 0.8001 | Val Loss: 0.4329, Val Acc: 0.7930
EarlyStopping counter: 2 out of 10
Epoch 60/100 | Train Loss: 0.4308, Train Acc: 0.7931 | Val Loss: 0.4322, Val Acc: 0.8000
EarlyStopping counter: 3 out of 10
Epoch 61/100 | Train Loss: 0.4371, Train Acc: 0.7895 | Val Loss: 0.4297, Val Acc: 0.7985
Validation loss decreased. Saving model to /home/pxk409/Skin_EfficientNet/Benchmark_GoogLeNet_CV_fold_1.pth
Epoch 62/100 | Train Loss: 0.4368, Train Acc: 0.7913 | Val Loss: 0.4336, Val Acc: 0.8040
EarlyStopping counter: 1 out of 10
Epoch 63/100 | Train Loss: 0.4427, Train Acc: 0.7884 | Val Loss: 0.4298, Val Acc: 0.7965
EarlyStopping counter

Epoch 16/100 | Train Loss: 0.4885, Train Acc: 0.7606 | Val Loss: 0.4668, Val Acc: 0.7790
EarlyStopping counter: 3 out of 10
Epoch 17/100 | Train Loss: 0.4773, Train Acc: 0.7680 | Val Loss: 0.4561, Val Acc: 0.7860
Validation loss decreased. Saving model to /home/pxk409/Skin_EfficientNet/Benchmark_GoogLeNet_CV_fold_2.pth
Epoch 18/100 | Train Loss: 0.4688, Train Acc: 0.7779 | Val Loss: 0.4636, Val Acc: 0.7805
EarlyStopping counter: 1 out of 10
Epoch 19/100 | Train Loss: 0.4823, Train Acc: 0.7762 | Val Loss: 0.4615, Val Acc: 0.7805
EarlyStopping counter: 2 out of 10
Epoch 20/100 | Train Loss: 0.4734, Train Acc: 0.7730 | Val Loss: 0.4646, Val Acc: 0.7895
EarlyStopping counter: 3 out of 10
Epoch 21/100 | Train Loss: 0.4734, Train Acc: 0.7705 | Val Loss: 0.4592, Val Acc: 0.7780
EarlyStopping counter: 4 out of 10
Epoch 22/100 | Train Loss: 0.4704, Train Acc: 0.7638 | Val Loss: 0.4593, Val Acc: 0.7860
EarlyStopping counter: 5 out of 10
Epoch 23/100 | Train Loss: 0.4677, Train Acc: 0.7760 | Val 

Epoch 36/100 | Train Loss: 0.4566, Train Acc: 0.7788 | Val Loss: 0.4770, Val Acc: 0.7795
EarlyStopping counter: 6 out of 10
Epoch 37/100 | Train Loss: 0.4542, Train Acc: 0.7825 | Val Loss: 0.4546, Val Acc: 0.7945
EarlyStopping counter: 7 out of 10
Epoch 38/100 | Train Loss: 0.4587, Train Acc: 0.7790 | Val Loss: 0.4575, Val Acc: 0.7900
EarlyStopping counter: 8 out of 10
Epoch 39/100 | Train Loss: 0.4673, Train Acc: 0.7749 | Val Loss: 0.4506, Val Acc: 0.7920
Validation loss decreased. Saving model to /home/pxk409/Skin_EfficientNet/Benchmark_GoogLeNet_CV_fold_3.pth
Epoch 40/100 | Train Loss: 0.4469, Train Acc: 0.7819 | Val Loss: 0.4645, Val Acc: 0.7825
EarlyStopping counter: 1 out of 10
Epoch 41/100 | Train Loss: 0.4520, Train Acc: 0.7818 | Val Loss: 0.4530, Val Acc: 0.7960
EarlyStopping counter: 2 out of 10
Epoch 42/100 | Train Loss: 0.4494, Train Acc: 0.7824 | Val Loss: 0.4594, Val Acc: 0.7905
EarlyStopping counter: 3 out of 10
Epoch 43/100 | Train Loss: 0.4528, Train Acc: 0.7815 | Val 

Epoch 29/100 | Train Loss: 0.4708, Train Acc: 0.7746 | Val Loss: 0.4545, Val Acc: 0.7920
EarlyStopping counter: 3 out of 10
Epoch 30/100 | Train Loss: 0.4597, Train Acc: 0.7755 | Val Loss: 0.4427, Val Acc: 0.8040
EarlyStopping counter: 4 out of 10
Epoch 31/100 | Train Loss: 0.4664, Train Acc: 0.7772 | Val Loss: 0.4417, Val Acc: 0.7940
Validation loss decreased. Saving model to /home/pxk409/Skin_EfficientNet/Benchmark_GoogLeNet_CV_fold_4.pth
Epoch 32/100 | Train Loss: 0.4599, Train Acc: 0.7762 | Val Loss: 0.4421, Val Acc: 0.7990
EarlyStopping counter: 1 out of 10
Epoch 33/100 | Train Loss: 0.4600, Train Acc: 0.7782 | Val Loss: 0.4360, Val Acc: 0.7955
Validation loss decreased. Saving model to /home/pxk409/Skin_EfficientNet/Benchmark_GoogLeNet_CV_fold_4.pth
Epoch 34/100 | Train Loss: 0.4631, Train Acc: 0.7728 | Val Loss: 0.4411, Val Acc: 0.7920
EarlyStopping counter: 1 out of 10
Epoch 35/100 | Train Loss: 0.4539, Train Acc: 0.7839 | Val Loss: 0.4439, Val Acc: 0.7950
EarlyStopping counter

Epoch 24/100 | Train Loss: 0.4670, Train Acc: 0.7742 | Val Loss: 0.4560, Val Acc: 0.7865
EarlyStopping counter: 4 out of 10
Epoch 25/100 | Train Loss: 0.4659, Train Acc: 0.7751 | Val Loss: 0.4580, Val Acc: 0.7915
EarlyStopping counter: 5 out of 10
Epoch 26/100 | Train Loss: 0.4683, Train Acc: 0.7628 | Val Loss: 0.4591, Val Acc: 0.7885
EarlyStopping counter: 6 out of 10
Epoch 27/100 | Train Loss: 0.4606, Train Acc: 0.7799 | Val Loss: 0.4517, Val Acc: 0.7915
EarlyStopping counter: 7 out of 10
Epoch 28/100 | Train Loss: 0.4612, Train Acc: 0.7772 | Val Loss: 0.4702, Val Acc: 0.7725
EarlyStopping counter: 8 out of 10
Epoch 29/100 | Train Loss: 0.4609, Train Acc: 0.7741 | Val Loss: 0.4503, Val Acc: 0.7910
EarlyStopping counter: 9 out of 10
Epoch 30/100 | Train Loss: 0.4562, Train Acc: 0.7722 | Val Loss: 0.4475, Val Acc: 0.7915
Validation loss decreased. Saving model to /home/pxk409/Skin_EfficientNet/Benchmark_GoogLeNet_CV_fold_5.pth
Epoch 31/100 | Train Loss: 0.4551, Train Acc: 0.7779 | Val 

Epoch 82/100 | Train Loss: 0.4090, Train Acc: 0.8067 | Val Loss: 0.4297, Val Acc: 0.8065
EarlyStopping counter: 1 out of 10
Epoch 83/100 | Train Loss: 0.4224, Train Acc: 0.7980 | Val Loss: 0.4305, Val Acc: 0.8000
EarlyStopping counter: 2 out of 10
Epoch 84/100 | Train Loss: 0.4207, Train Acc: 0.8011 | Val Loss: 0.4481, Val Acc: 0.7860
EarlyStopping counter: 3 out of 10
Epoch 85/100 | Train Loss: 0.4178, Train Acc: 0.8015 | Val Loss: 0.4300, Val Acc: 0.7960
EarlyStopping counter: 4 out of 10
Epoch 86/100 | Train Loss: 0.4089, Train Acc: 0.8071 | Val Loss: 0.4299, Val Acc: 0.8010
EarlyStopping counter: 5 out of 10
Epoch 87/100 | Train Loss: 0.4195, Train Acc: 0.7995 | Val Loss: 0.4287, Val Acc: 0.8025
EarlyStopping counter: 6 out of 10
Epoch 88/100 | Train Loss: 0.4119, Train Acc: 0.7995 | Val Loss: 0.4330, Val Acc: 0.7885
EarlyStopping counter: 7 out of 10
Epoch 89/100 | Train Loss: 0.4181, Train Acc: 0.8001 | Val Loss: 0.4523, Val Acc: 0.7855
EarlyStopping counter: 8 out of 10
Epoch 90

In [31]:
# Convert results to DataFrame
cv_metrics_df = pd.DataFrame(cv_metrics)

# Calculate average metrics
avg_metrics = cv_metrics_df.mean().to_dict()
avg_metrics["Fold"] = "Average"

# Add the average metrics to the DataFrame
cv_metrics_df = cv_metrics_df.append(avg_metrics, ignore_index=True)

# Save results to CSV file
csv_file_name = f"CV_Results/{file_name}.csv"
cv_metrics_df.to_csv(csv_file_name, index=False)

print(f"Cross-validation metrics saved to {csv_file_name}")

Cross-validation metrics saved to CV_Results/Benchmark_GoogLeNet_CV.csv


/tmp/job.2295042.hpc/ipykernel_4142477/1908723448.py:9: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  cv_metrics_df = cv_metrics_df.append(avg_metrics, ignore_index=True)
